<a href="https://colab.research.google.com/github/jingyig16/ufo-sightings/blob/main/ufo_sightings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CIS 5450 Final Project - UFO Shape Prediction**

# Part 1: Introduction

(todo)

# Part 2: Data Loading & Preprocessing

In [ ]:
import pandas as pd
import re
import reverse_geocoder as rg

## The UFO Sighting Dataset

###Information of The Raw Data

In [ ]:
# Data loading via GitHub
github_url = 'https://raw.githubusercontent.com/jingyig16/ufo-sightings/refs/heads/main/ufo_sighting.csv'
ufo_df = pd.read_csv(github_url, low_memory=False)

In [ ]:
ufo_df

,datetime,city,state,country,shape,duration (seconds),duration (hours/min),comments,date posted,latitude,longitude
0,10/10/1949 20:30,san marcos,tx,us,cylinder,2700,45 minutes,This event took place in early fall around 194...,4/27/2004,29.8830556,-97.941111
1,10/10/1949 21:00,lackland afb,tx,NaN,light,7200,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,12/16/2005,29.38421,-98.581082
2,10/10/1955 17:00,chester (uk/england),NaN,gb,circle,20,20 seconds,Green/Orange circular disc over Chester&#44 En...,1/21/2008,53.2,-2.916667
3,10/10/1956 21:00,edna,tx,us,circle,20,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.9783333,-96.645833
4,10/10/1960 20:00,kaneohe,hi,us,light,900,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.4180556,-157.803611
...,...,...,...,...,...,...,...,...,...,...,...
80327,9/9/2013 21:15,nashville,tn,us,light,600,10 minutes,Round from the distance/slowly changing colors...,9/30/2013,36.1658333,-86.784444
80328,9/9/2013 22:00,boise,id,us,circle,1200,20 minutes,Boise&#44 ID&#44 spherical&#44 20 min&#44 10 r...,9/30/2013,43.6136111,-116.202500
80329,9/9/2013 22:00,napa,ca,us,other,1200,hour,Napa UFO&#44,9/30/2013,38.2972222,-122.284444
80330,9/9/2013 22:20,vienna,va,us,circle,5,5 seconds,Saw a five gold lit cicular craft moving fastl...,9/30/2013,38.9011111,-77.265556


In [ ]:
ufo_df.columns = ufo_df.columns.str.strip().str.lower()
for i, col in enumerate(ufo_df.columns):
    print(f"{i+1}. {col}")

1. datetime
2. city
3. state
4. country
5. shape
6. duration (seconds)
7. duration (hours/min)
8. comments
9. date posted
10. latitude
11. longitude


In [ ]:
len(ufo_df)

80332

In [ ]:
ufo_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80332 entries, 0 to 80331
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   datetime              80332 non-null  object 
 1   city                  80332 non-null  object 
 2   state                 74535 non-null  object 
 3   country               70662 non-null  object 
 4   shape                 78400 non-null  object 
 5   duration (seconds)    80332 non-null  object 
 6   duration (hours/min)  80332 non-null  object 
 7   comments              80317 non-null  object 
 8   date posted           80332 non-null  object 
 9   latitude              80332 non-null  object 
 10  longitude             80332 non-null  float64
dtypes: float64(1), object(10)
memory usage: 6.7+ MB


### Data Processing

#### Handling Redundant and Invalid Data

Since the UFO `shape` is our target variable for prediction, we removed all records where this field was missing. Supervised learning models require a known label for each training example in order to compute loss and evaluate performance. Instances without a recorded shape cannot contribute to the learning process and would otherwise introduce noise or bias into the model. Therefore, rows with null values in the shape column were excluded from the dataset.

In [ ]:
ufo_df = ufo_df.dropna(subset=['shape'])

In [ ]:
len(ufo_df)

78400

In [ ]:
ufo_cleaned = ufo_df.copy()

##### Drop Unused Columns

In [ ]:
# Drop unused columns
ufo_cleaned = ufo_cleaned.drop(columns=["comments", "date posted","duration (hours/min)"])

The `duration (hours/min)` column was dropped because it duplicates the information in duration (seconds) but in inconsistent units. Keeping only the numeric duration simplifies analysis and ensures accurate quantitative modeling.

The `comments` column was removed since it contains unstructured text outside the scope of this study, which focuses on numerical and spatial patterns rather than linguistic content.

The `date posted` column was excluded because it reflects when a report was submitted, not when the sighting occurred. The true event timing is already captured in the `datetime` field, making date posted less meaningful for UFO pattern analysis.

#### Convert `datetime` to Datetime type

Converting the datetime column to datetime format ensures consistent and accurate time-based analysis and operations across the dataset.

We found that some records in `datetime` contained timestamps using "24:00", which is invalid in Python's datetime format. The value "24:00" represents midnight of the following day, but since most libraries only accept hours from 00 to 23, these entries caused conversion errors. To handle this, all "24:00" values were replaced with "00:00"

In [ ]:
ufo_cleaned["datetime"] = (ufo_cleaned["datetime"].astype(str).apply(lambda x: re.sub(r"24:00", "00:00", x)))
ufo_cleaned["datetime"] = pd.to_datetime(ufo_cleaned["datetime"])

This small adjustment preserves the temporal meaning of the records while ensuring compatibility with pandas datetime operations.

#### Standardize `latitude` and `longitude`

In [ ]:
def extract_numeric(value):
    """
    Extracts the first valid float-like number (with optional minus sign and decimal)
    from a string. Returns NaN if no valid number found.
    """
    if pd.isna(value):
        return None
    match = re.search(r"-?\d+\.?\d*", str(value))
    if match:
        return float(match.group())
    return None

In [ ]:
ufo_cleaned["latitude"] = ufo_cleaned["latitude"].apply(extract_numeric)
ufo_cleaned["longitude"] = ufo_cleaned["longitude"].apply(extract_numeric)
# Convert to numeric just in case and drop invalid rows
ufo_cleaned["latitude"] = pd.to_numeric(ufo_cleaned["latitude"], errors="coerce")
ufo_cleaned["longitude"] = pd.to_numeric(ufo_cleaned["longitude"], errors="coerce")

In [ ]:
ufo_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 78400 entries, 0 to 80331
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   datetime            78400 non-null  datetime64[ns]
 1   city                78400 non-null  object        
 2   state               72742 non-null  object        
 3   country             69001 non-null  object        
 4   shape               78400 non-null  object        
 5   duration (seconds)  78400 non-null  object        
 6   latitude            78400 non-null  float64       
 7   longitude           78400 non-null  float64       
dtypes: datetime64[ns](1), float64(2), object(5)
memory usage: 5.4+ MB


To ensure all coordinate information was standardized for downstream geospatial analysis, I implemented a custom cleaning pipeline using a regex-based function:*extract_numeric* to extract valid float-like numbers from potentially messy latitude and longitude entries. This approach handled irregular formats (e.g., values containing symbols, extra spaces, or text) and safely converted all coordinates to numeric types. After cleaning, both latitude and longitude columns contained 78,400 valid float64 values, confirming that all coordinate data were successfully retained and standardized.

#### Standardize `city`, `state`, and `country`

In [ ]:
!pip install reverse_geocoder

In [ ]:
# Extract coordinates
coords = list(zip(ufo_cleaned["latitude"], ufo_cleaned["longitude"]))
# Batch reverse-geocode
results = rg.search(coords)
# Create geocoded DataFrame and align
geo_df = pd.DataFrame(results)
geo_df.index = ufo_cleaned.index
# Replace state & country with standardized values
ufo_cleaned["city"] = geo_df["name"]
ufo_cleaned["state"] = geo_df["admin1"]
ufo_cleaned["country"] = geo_df["cc"]

Loading formatted geocoded file...


We regenerated the `city`, `state`, and `country` columns directly from the latitude and longitude coordinates to achieve a consistent and standardized geographic representation across all records. The original text-based fields contained irregularities such as inconsistent capitalization, abbreviations, and mixed naming conventions, which made them unreliable for quantitative analysis or model training. In contrast, latitude and longitude are precise and continuous numerical values that can be used to objectively infer standardized location information through reverse geocoding. This ensures that entries with identical coordinates map to the same administrative regions, eliminating human errors and variations in naming.
